# AD 698 — Section-Scoped RAG over SEC 10-K (2024)
**Domain Track:** Financial Performance & Risk Analysis
**SEC 10-K Items in scope:** Item 6 (Selected Financial Data), Item 7 (MD&A), Item 7A (Market Risk), Item 8 (Financial Statements)
**Target corpus:** 15–20 firms, FY 2024 10-K filings (HTML)
**Source:** Google Drive folder `SEC-10K-2024-HTML` (mounted in Colab)

---

This notebook implements **all five milestones** of the project end-to-end:

| Milestone | Deliverable |
|---|---|
| M1 | Section-scoped extraction + cleaning (Items 6/7/7A/8 only) |
| M2 | Token-aware chunking + structured metadata |
| M3 | Embeddings + FAISS with strict section filtering |
| M4 | Grounded RAG: retrieval → context → LLM → cited JSON answer |
| M5 | Evaluation: Recall@k, citation coverage, hallucination estimate, leakage check |

> **How to run:** Runtime → Run all. The notebook caches intermediate artifacts to `/content/drive/MyDrive/AD698_cache/` so subsequent runs are fast. Each milestone section is self-contained and can be re-run independently after its dependencies are cached.


## 0 · Environment Setup


### 0.1 Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### 0.2 Install dependencies


In [ ]:
!pip -q install sentence-transformers==3.0.1 \
    faiss-cpu==1.8.0 \
    tiktoken==0.7.0 \
    beautifulsoup4==4.12.3 \
    lxml==5.2.2 \
    pandas==2.2.2 \
    rapidfuzz==3.9.6 \
    openai==1.40.0 \
    google-generativeai==0.7.2 \
    tqdm


### 0.3 Imports


In [ ]:
import os, re, json, glob, hashlib, unicodedata, html, time
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict, field
from typing import Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from bs4 import BeautifulSoup
import tiktoken
from sentence_transformers import SentenceTransformer
import faiss

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)


### 0.4 Configuration


In [ ]:
# --- Paths (edit only if your Drive layout differs) ---
DRIVE_ROOT       = '/content/drive/MyDrive'
FILINGS_DIR      = f'{DRIVE_ROOT}/SEC-10K-2024-HTML'   # input: raw 10-K HTML
CACHE_DIR        = f'{DRIVE_ROOT}/AD698_cache'         # intermediate artifacts
OUTPUTS_DIR      = f'{DRIVE_ROOT}/AD698_outputs'       # final results

for d in (CACHE_DIR, OUTPUTS_DIR):
    os.makedirs(d, exist_ok=True)

# --- Domain scope: Financial Performance & Risk ---
# These are the ONLY SEC 10-K Items our RAG system will retrieve over.
ALLOWED_ITEMS = ['Item 6', 'Item 7', 'Item 7A', 'Item 8']

# --- Chunking ---
CHUNK_TOKENS   = 500   # target tokens per chunk
CHUNK_OVERLAP  = 50    # overlap between consecutive chunks
TOKEN_ENCODER  = 'cl100k_base'  # tiktoken encoding (GPT-3.5/4 compatible)

# --- Embeddings ---
EMBED_MODEL    = 'BAAI/bge-small-en-v1.5'  # free, open, 384-dim, strong on financial text
EMBED_DIM      = 384

# --- Retrieval ---
TOP_K          = 5
MIN_SIM        = 0.20   # below this, treat as "no relevant evidence"

# --- Smoke-test knob ---
# Set MAX_FILES to a small int (e.g. 3) to run end-to-end on a subset.
# Leave as None for the full corpus.
MAX_FILES      = None

# --- LLM backend for generation (M4) ---
# Pick ONE of: 'openai', 'gemini', 'mock'
# 'mock' = deterministic stub answer, no API key needed — use it for a dry run.
LLM_BACKEND    = 'mock'
OPENAI_MODEL   = 'gpt-4o-mini'
GEMINI_MODEL   = 'gemini-1.5-flash'

# API keys (preferred: Colab Secrets → left sidebar → 🔑 → add OPENAI_API_KEY / GEMINI_API_KEY)
try:
    from google.colab import userdata
    os.environ.setdefault('OPENAI_API_KEY', userdata.get('OPENAI_API_KEY') or '')
    os.environ.setdefault('GEMINI_API_KEY', userdata.get('GEMINI_API_KEY') or '')
except Exception:
    pass

print('Config loaded.')
print(f'  Filings dir : {FILINGS_DIR}')
print(f'  Cache dir   : {CACHE_DIR}')
print(f'  Allowed items: {ALLOWED_ITEMS}')
print(f'  LLM backend : {LLM_BACKEND}')


### 0.5 Scope Boundaries (read first)

> **In scope:** narrative financial Q&A over SEC 10-K Items 6, 7, 7A, 8 — revenue drivers, liquidity, market risk, accounting policy.
>
> **Out of scope (by design):**
> - **Precise numeric/table lookups** (e.g., "What was Q4 EPS?") — Item 8 tables are replaced with `[TABLE]` placeholders; a future iteration should index tables separately.
> - **Cross-firm comparisons** — each answer is firm-scoped; aggregation is an analytics-layer concern, not RAG.
> - **Governance / compensation / cyber / forward guidance beyond the filing** — outside the Allow-List (Items 6/7/7A/8). Enforced by **post-retrieval hard filter**, not prompt instruction.
>
> See `docs/SCOPE_BOUNDARIES.md` for the full rationale and enforcement mechanism.


---
## Milestone 1 · Section Extraction & Cleaning

**Goal:** For every 10-K HTML in Drive, isolate the text of **Items 6, 7, 7A, 8** and strip TOC bleed, repeated headers, signature blocks, and HTML artifacts. Output: clean, section-scoped JSONL.


### 1.1 Enumerate filings in Drive


In [ ]:
def list_filings(filings_dir: str) -> list[str]:
    if not os.path.isdir(filings_dir):
        raise FileNotFoundError(
            f'Filings dir not found: {filings_dir}\n'
            f"Expected folder 'SEC-10K-2024-HTML' in the root of your MyDrive. "
            f"If the folder was shared with you, right-click it in drive.google.com "
            f"→ 'Add shortcut to Drive' → choose MyDrive."
        )
    patterns = ('*.htm', '*.html', '*.HTM', '*.HTML')
    paths = []
    for p in patterns:
        paths.extend(glob.glob(os.path.join(filings_dir, '**', p), recursive=True))
    paths = sorted(set(paths))
    return paths

FILING_PATHS = list_filings(FILINGS_DIR)
if MAX_FILES is not None:
    FILING_PATHS = FILING_PATHS[:MAX_FILES]
    print(f'[smoke-test] MAX_FILES={MAX_FILES} — using only first {len(FILING_PATHS)} filings')
print(f'Found {len(FILING_PATHS)} HTML filings.')
for p in FILING_PATHS[:5]:
    print(' -', os.path.basename(p))
if len(FILING_PATHS) > 5:
    print(f' ... and {len(FILING_PATHS)-5} more')


### 1.2 Parse filing header → CIK, company, ticker, filing year


In [ ]:
# SEC conforming HTML headers have 'CENTRAL INDEX KEY', 'COMPANY CONFORMED NAME',
# 'TRADING SYMBOL', 'CONFORMED PERIOD OF REPORT'. We also back off to filename.

HEADER_PATTERNS = {
    'cik':     re.compile(r'CENTRAL\s*INDEX\s*KEY[^\d]*(\d{4,10})', re.I),
    'company': re.compile(r'COMPANY\s*CONFORMED\s*NAME\s*[:=]?\s*([A-Z0-9 ,.\-&/]+)', re.I),
    'ticker':  re.compile(r'TRADING\s*SYMBOL\s*[:=]?\s*([A-Z]{1,6})', re.I),
    'period':  re.compile(r'CONFORMED\s*PERIOD\s*OF\s*REPORT\s*[:=]?\s*(\d{8})', re.I),
}

def extract_header(raw_text: str, filename: str) -> dict:
    hdr = {'cik': None, 'company': None, 'ticker': None, 'filing_year': None}
    for key, pat in HEADER_PATTERNS.items():
        m = pat.search(raw_text[:50000])  # header is at top
        if m:
            val = m.group(1).strip()
            if key == 'period':
                hdr['filing_year'] = val[:4]
            else:
                hdr[key] = val
    # Fallbacks from filename like 'AAPL_10-K_2024.htm' or 'cik0000320193-20240928.htm'
    base = os.path.splitext(os.path.basename(filename))[0]
    if not hdr['ticker']:
        m = re.match(r'^([A-Z]{1,6})[_\-.]', base)
        if m: hdr['ticker'] = m.group(1)
    if not hdr['filing_year']:
        m = re.search(r'(20\d{2})', base)
        if m: hdr['filing_year'] = m.group(1)
    if not hdr['cik']:
        m = re.search(r'cik[-_]?0*(\d{4,10})', base, re.I)
        if m: hdr['cik'] = m.group(1)
    return hdr


### 1.3 HTML → plain text, preserving item-boundary markers


In [ ]:
# Strategy:
# 1. Parse with BeautifulSoup (lxml).
# 2. Drop <script>, <style>, <table> INSIDE items kept separately for M8 financial
#    tables — for text extraction we convert tables to ' [TABLE] ' placeholders so
#    they don't flood retrieval with numeric noise.
# 3. Collapse whitespace but keep line breaks around headings so item regex works.

TABLE_PLACEHOLDER = ' [TABLE] '

def html_to_text(raw_html: str) -> str:
    soup = BeautifulSoup(raw_html, 'lxml')
    # kill non-content
    for t in soup(['script', 'style', 'head', 'meta', 'link']):
        t.decompose()
    # replace tables with placeholder (we keep financial tables in M8 separately)
    for tbl in soup.find_all('table'):
        tbl.replace_with(TABLE_PLACEHOLDER)
    # add newlines after block elements so item regex sees line breaks
    for tag in soup.find_all(['p', 'div', 'br', 'h1','h2','h3','h4','h5','h6', 'li']):
        tag.append('\n')
    text = soup.get_text('\n', strip=False)
    # unicode / html entity normalization
    text = unicodedata.normalize('NFKC', text)
    text = html.unescape(text)
    # collapse 3+ blank lines and runs of whitespace on each line
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n[ \t]+', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


### 1.4 Item boundary detection (6, 7, 7A, 8) with TOC-bleed filtering


In [ ]:
# The regex must match real item headings but SKIP table-of-contents hits.
# Heuristic: a real heading line is usually followed by substantial body text
# (> 200 chars) before the NEXT item heading; a TOC entry is followed by a page
# number / another TOC line within ~100 chars.

# Full ordered list of items so we know what a "next item" looks like.
ITEM_SEQUENCE = [
    'Item 1', 'Item 1A', 'Item 1B', 'Item 1C', 'Item 2', 'Item 3', 'Item 4',
    'Item 5', 'Item 6', 'Item 7', 'Item 7A', 'Item 8', 'Item 9', 'Item 9A',
    'Item 9B', 'Item 9C', 'Item 10', 'Item 11', 'Item 12', 'Item 13', 'Item 14',
    'Item 15', 'Item 16',
]

def item_regex(item: str) -> re.Pattern:
    # Match 'Item 7.', 'ITEM 7 ', 'Item 7 —', case-insensitive, at line start.
    num = item.split()[1]  # '7', '7A', etc.
    return re.compile(
        rf'(?im)^\s*item\s+{re.escape(num)}\b\s*[\.\-—:]?\s*'
    )

def find_all_item_spans(text: str) -> list[dict]:
    'Return every item-heading hit with its position and the following 400 chars for inspection.'
    hits = []
    for item in ITEM_SEQUENCE:
        for m in item_regex(item).finditer(text):
            start = m.start()
            hits.append({'item': item, 'start': start, 'preview': text[start:start+200]})
    hits.sort(key=lambda h: h['start'])
    return hits

def filter_toc_hits(hits: list[dict], text: str, min_body: int = 400) -> list[dict]:
    '''Keep only hits where the body to the next hit is long enough to be real prose.
    TOC hits are clustered (next hit very close) -- those get dropped.
    '''
    keep = []
    for i, h in enumerate(hits):
        next_start = hits[i+1]['start'] if i+1 < len(hits) else len(text)
        body_len = next_start - h['start']
        if body_len >= min_body:
            keep.append(h)
    return keep

def extract_items(text: str, wanted: list[str]) -> dict[str, str]:
    '''Return {item_name: body_text} for each wanted item, using the FIRST real
    (post-TOC-filter) occurrence of each item heading.'''
    all_hits  = find_all_item_spans(text)
    real_hits = filter_toc_hits(all_hits, text)
    # For each wanted item, take its first real hit and slice until the next real hit.
    sections = {}
    for i, h in enumerate(real_hits):
        if h['item'] not in wanted:
            continue
        if h['item'] in sections:
            continue  # already captured the first real instance
        end = real_hits[i+1]['start'] if i+1 < len(real_hits) else len(text)
        sections[h['item']] = text[h['start']:end].strip()
    return sections


### 1.5 Per-section cleaning


In [ ]:
SIGNATURE_BLOCK = re.compile(
    r'(?is)(signatures?\s*\n.+?pursuant\s+to\s+the\s+requirements.+?$)'
)
BOILERPLATE_PATTERNS = [
    # Repeated page headers like 'Apple Inc. | 2024 Form 10-K | Page 12'
    re.compile(r'(?im)^\s*[A-Z][A-Za-z0-9 ,.&\-]+\|\s*20\d{2}\s*Form\s*10-K\s*\|\s*Page\s*\d+\s*$'),
    # Standalone page numbers on their own line
    re.compile(r'(?m)^\s*\d{1,3}\s*$'),
    # 'Table of Contents' header that sometimes leaks in
    re.compile(r'(?im)^\s*table\s+of\s+contents\s*$'),
]

def clean_section(txt: str) -> str:
    # strip signature block if present
    txt = SIGNATURE_BLOCK.sub('', txt)
    for pat in BOILERPLATE_PATTERNS:
        txt = pat.sub('', txt)
    # drop '[TABLE]' noise collapse (keep as single marker)
    txt = re.sub(r'(\[TABLE\]\s*){2,}', '[TABLE] ', txt)
    # collapse repeated blank lines
    txt = re.sub(r'\n{3,}', '\n\n', txt)
    # strip trailing whitespace on each line
    txt = '\n'.join(line.rstrip() for line in txt.splitlines())
    return txt.strip()


### 1.6 Run extraction over the whole corpus


In [ ]:
SECTIONS_PATH = f'{CACHE_DIR}/sections.jsonl'

def build_sections(filings: list[str], wanted: list[str]) -> list[dict]:
    rows = []
    for path in tqdm(filings, desc='Extracting items'):
        with open(path, 'rb') as f:
            raw = f.read().decode('utf-8', errors='replace')
        hdr = extract_header(raw, path)
        text = html_to_text(raw)
        sections = extract_items(text, wanted)
        for item, body in sections.items():
            clean = clean_section(body)
            if len(clean) < 500:
                continue  # skip items that are placeholders like 'Not applicable.'
            rows.append({
                **hdr,
                'file': os.path.basename(path),
                'item': item,
                'char_count': len(clean),
                'text': clean,
            })
    return rows

SECTION_ROWS = build_sections(FILING_PATHS, ALLOWED_ITEMS)

with open(SECTIONS_PATH, 'w') as f:
    for r in SECTION_ROWS:
        f.write(json.dumps(r) + '\n')

print(f'Wrote {len(SECTION_ROWS)} section records to {SECTIONS_PATH}')


### 1.7 Extraction coverage report


In [ ]:
if SECTION_ROWS:
    df_sec = pd.DataFrame(SECTION_ROWS)
    coverage = (df_sec.groupby(['company', 'item'])['char_count']
                      .first().unstack(fill_value=0))
    print('Characters of clean text per firm × item (zeros = section missing or too short):')
    display(coverage)
    missing = []
    for company, row in coverage.iterrows():
        for item in ALLOWED_ITEMS:
            if item not in coverage.columns or row.get(item, 0) == 0:
                missing.append((company, item))
    if missing:
        print(f'\nWARNING: {len(missing)} (firm, item) pairs missing or under 500 chars.')
        print('First 10:', missing[:10])
else:
    print('No sections extracted. Check FILINGS_DIR and filename patterns.')


---
## Milestone 2 · Token-Aware Chunking + Metadata Engineering

**Why tokens not characters:** downstream models (embeddings + generation) bill and bound by tokens. A 500-char chunk could be 80 or 200 tokens depending on Item-8 number density — retrieval quality collapses either way.

**Strategy:** split each Item body into ~500-token windows with 50-token overlap, breaking on sentence boundaries where possible. Attach full provenance metadata.


### 2.1 Token-aware chunker


In [ ]:
ENC = tiktoken.get_encoding(TOKEN_ENCODER)

SENT_SPLIT = re.compile(r'(?<=[\.\?\!])\s+(?=[A-Z\(\[])')

def tok_len(s: str) -> int:
    return len(ENC.encode(s))

def split_into_sentences(text: str) -> list[str]:
    # Keep paragraph structure: split each paragraph on sentence ends, flatten.
    sents = []
    for para in text.split('\n\n'):
        para = para.strip()
        if not para:
            continue
        parts = SENT_SPLIT.split(para)
        sents.extend(p.strip() for p in parts if p.strip())
    return sents

def chunk_text(text: str,
               target_tokens: int = CHUNK_TOKENS,
               overlap_tokens: int = CHUNK_OVERLAP) -> list[str]:
    sents = split_into_sentences(text)
    chunks, cur, cur_tok = [], [], 0
    for s in sents:
        st = tok_len(s)
        if st > target_tokens:
            # sentence itself is too long — fall back to token windowing
            tok_ids = ENC.encode(s)
            for i in range(0, len(tok_ids), target_tokens - overlap_tokens):
                chunks.append(ENC.decode(tok_ids[i:i+target_tokens]))
            continue
        if cur_tok + st > target_tokens and cur:
            chunks.append(' '.join(cur))
            # carry the tail for overlap
            tail, tail_tok = [], 0
            for t in reversed(cur):
                tlen = tok_len(t)
                if tail_tok + tlen > overlap_tokens:
                    break
                tail.insert(0, t)
                tail_tok += tlen
            cur, cur_tok = tail, tail_tok
        cur.append(s)
        cur_tok += st
    if cur:
        chunks.append(' '.join(cur))
    return chunks


### 2.2 Run chunking + attach metadata


In [ ]:
CHUNKS_PATH = f'{CACHE_DIR}/chunks.jsonl'

def build_chunks(section_rows: list[dict]) -> list[dict]:
    out = []
    for r in tqdm(section_rows, desc='Chunking'):
        for i, piece in enumerate(chunk_text(r['text'])):
            h = hashlib.md5(f"{r.get('cik')}|{r['item']}|{i}|{piece[:40]}".encode()).hexdigest()[:10]
            chunk_id = f"{(r.get('ticker') or r.get('cik') or 'UNK')}-{r['item'].replace(' ','')}-{i:03d}-{h}"
            out.append({
                'chunk_id'    : chunk_id,
                'cik'         : r.get('cik'),
                'company'     : r.get('company'),
                'ticker'      : r.get('ticker'),
                'filing_year' : r.get('filing_year'),
                'item'        : r['item'],
                'chunk_index' : i,
                'token_count' : tok_len(piece),
                'text'        : piece,
            })
    return out

CHUNKS = build_chunks(SECTION_ROWS)

with open(CHUNKS_PATH, 'w') as f:
    for c in CHUNKS:
        f.write(json.dumps(c) + '\n')

print(f'Wrote {len(CHUNKS):,} chunks → {CHUNKS_PATH}')


### 2.3 Chunk distribution analysis


In [ ]:
df_chunks = pd.DataFrame(CHUNKS)
print('Token-count stats per Item:')
display(df_chunks.groupby('item')['token_count'].describe().round(1))

print('\nChunk count per firm × item:')
display(df_chunks.pivot_table(index='company', columns='item',
                              values='chunk_id', aggfunc='count', fill_value=0))

# Quick sanity: any chunks way over target?
over = df_chunks[df_chunks.token_count > CHUNK_TOKENS * 1.2]
print(f'\n{len(over)} chunks exceed 120% of target token budget (should be ≈0).')


---
## Milestone 3 · Embedding Layer + Section-Scoped Retrieval

**Why BGE-small:** strong on finance/legal text in the MTEB benchmark, 384-dim (index stays small), and fully open (no API key). Swap to `text-embedding-3-small` if you have an OpenAI key — the retrieval interface is identical.

**Section scoping** is enforced at query time, not by partitioning the index. A single FAISS index holds all chunks; the `allowed_items` filter excludes any candidate whose metadata Item is outside the allow-list. This gives correct scope enforcement while keeping the index unified.


### 3.1 Embed all chunks


In [ ]:
EMB_PATH  = f'{CACHE_DIR}/embeddings.npy'
META_PATH = f'{CACHE_DIR}/chunk_meta.jsonl'

def embed_corpus(chunks: list[dict]) -> np.ndarray:
    model = SentenceTransformer(EMBED_MODEL)
    texts = [c['text'] for c in chunks]
    emb = model.encode(
        texts, batch_size=64, show_progress_bar=True,
        normalize_embeddings=True,  # so inner product == cosine
    )
    return np.asarray(emb, dtype='float32')

if os.path.exists(EMB_PATH) and os.path.exists(META_PATH):
    print('Loading cached embeddings…')
    embeddings = np.load(EMB_PATH)
    with open(META_PATH) as f:
        CHUNKS_META = [json.loads(l) for l in f]
else:
    embeddings = embed_corpus(CHUNKS)
    np.save(EMB_PATH, embeddings)
    # save metadata WITHOUT text so the index stays light; full text re-joins at retrieve time
    CHUNKS_META = [{k: v for k, v in c.items() if k != 'text'} for c in CHUNKS]
    with open(META_PATH, 'w') as f:
        for m in CHUNKS_META:
            f.write(json.dumps(m) + '\n')

print(f'Embeddings: {embeddings.shape}  dtype={embeddings.dtype}')


### 3.2 Build FAISS index


In [ ]:
INDEX_PATH = f'{CACHE_DIR}/faiss.index'

def build_faiss(emb: np.ndarray) -> faiss.Index:
    idx = faiss.IndexFlatIP(emb.shape[1])  # cosine (embeddings are normalized)
    idx.add(emb)
    return idx

if os.path.exists(INDEX_PATH):
    index = faiss.read_index(INDEX_PATH)
    print(f'Loaded FAISS index: {index.ntotal:,} vectors')
else:
    index = build_faiss(embeddings)
    faiss.write_index(index, INDEX_PATH)
    print(f'Built & saved FAISS index: {index.ntotal:,} vectors')

# in-memory text table for joining back after retrieval
CHUNK_BY_ID = {c['chunk_id']: c for c in CHUNKS}


### 3.3 Section-scoped retrieval


In [ ]:
EMBED_MODEL_OBJ = SentenceTransformer(EMBED_MODEL)

def retrieve(query: str,
             allowed_items: list[str] | None = None,
             k: int = TOP_K,
             over_fetch: int = 40) -> list[dict]:
    '''Return top-k chunks whose item ∈ allowed_items (default: ALLOWED_ITEMS).

    Returns list of {'score', 'chunk_id', 'item', 'company', 'ticker',
                     'cik', 'text'} — sorted by score desc.
    '''
    allowed = set(allowed_items or ALLOWED_ITEMS)
    q_vec = EMBED_MODEL_OBJ.encode([query], normalize_embeddings=True).astype('float32')
    # over-fetch then filter so we still return k after rejections
    sims, ids = index.search(q_vec, k * over_fetch if allowed else k)
    results = []
    for score, vec_id in zip(sims[0], ids[0]):
        if vec_id < 0:
            continue
        meta = CHUNKS_META[vec_id]
        if meta['item'] not in allowed:
            continue   # STRICT SECTION SCOPING — hard filter, no exceptions
        full = CHUNK_BY_ID[meta['chunk_id']]
        results.append({
            'score'    : float(score),
            'chunk_id' : meta['chunk_id'],
            'item'     : meta['item'],
            'company'  : meta['company'],
            'ticker'   : meta['ticker'],
            'cik'      : meta['cik'],
            'year'     : meta.get('filing_year'),
            'text'     : full['text'],
        })
        if len(results) >= k:
            break
    return results


### 3.4 Sanity check: zero cross-item leakage


In [ ]:
# Prove that retrieval never returns a chunk outside the allow-list,
# even for queries whose natural answer lives in a non-allowed item.

probes = [
    'board of directors independence and governance structure',   # naturally Item 10
    'executive compensation philosophy and pay-for-performance',  # naturally Item 11
    'risk factors related to cybersecurity and data breaches',    # naturally Item 1A
]
for q in probes:
    hits = retrieve(q, allowed_items=ALLOWED_ITEMS, k=5)
    items = {h['item'] for h in hits}
    leak = items - set(ALLOWED_ITEMS)
    status = 'LEAK' if leak else 'OK'
    print(f'[{status}] "{q[:60]}" → retrieved items={items}')

print('\nDemonstration query (expected to work — it is about MD&A):')
for h in retrieve('liquidity and capital resources discussion', k=3):
    print(f"  {h['score']:.3f}  {h['company']:<25}  {h['item']}  {h['text'][:120]}…")


---
## Milestone 4 · Grounded RAG Generation Engine

**Contract the system must honor:**
1. Output is **valid JSON** with fields `answer`, `citations[]`, `refused`.
2. Every factual claim in `answer` must trace to at least one entry in `citations` (each one = `{chunk_id, item}`).
3. If no retrieved chunk supports the question, set `refused=true` and explain in `answer`.
4. Do NOT answer from the model's parametric knowledge.


### 4.1 Prompt template


In [ ]:
RAG_SYSTEM_PROMPT = '''You are a careful financial analyst RAG system restricted to SEC 10-K 2024 filings.
Your job: answer the user\'s question USING ONLY the CONTEXT blocks below. Do NOT use outside knowledge.

Rules:
- If the context does not contain enough evidence, set "refused": true and explain what is missing.
- Every factual statement in your answer must be backed by at least one [chunk_id] in "citations".
- Cite the Item number (e.g. "Item 7") and the chunk_id next to each claim.
- Output MUST be strict JSON: {"answer": "...", "citations": [{"chunk_id": "...", "item": "..."}], "refused": false}.
- Do not invent chunk_ids. Only cite chunks that appear in the CONTEXT.
- Retrieved sections are limited to: Item 6, Item 7, Item 7A, Item 8. Refuse any question that requires other items.'''

def build_prompt(question: str, hits: list[dict]) -> tuple[str, str]:
    ctx_blocks = []
    for h in hits:
        ctx_blocks.append(
            f"[chunk_id={h['chunk_id']}] [item={h['item']}] "
            f"[company={h['company']}] [year={h['year']}]\n{h['text']}"
        )
    context = '\n\n---\n\n'.join(ctx_blocks)
    user = f"CONTEXT:\n{context}\n\nQUESTION: {question}\n\nRespond as strict JSON only."
    return RAG_SYSTEM_PROMPT, user


### 4.2 LLM clients (switchable: OpenAI / Gemini / mock)


In [ ]:
def _parse_json_answer(raw: str) -> dict:
    'Strip code fences and extract the JSON object.'
    raw = raw.strip()
    raw = re.sub(r'^```(?:json)?', '', raw).strip()
    raw = re.sub(r'```$', '', raw).strip()
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        raw = m.group(0)
    try:
        return json.loads(raw)
    except Exception as e:
        return {'answer': raw, 'citations': [], 'refused': True,
                'error': f'JSON parse failed: {e}'}

def llm_generate(system: str, user: str) -> dict:
    if LLM_BACKEND == 'openai':
        from openai import OpenAI
        client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
        resp = client.chat.completions.create(
            model=OPENAI_MODEL, temperature=0,
            response_format={'type': 'json_object'},
            messages=[
                {'role': 'system', 'content': system},
                {'role': 'user',   'content': user},
            ],
        )
        return _parse_json_answer(resp.choices[0].message.content)

    if LLM_BACKEND == 'gemini':
        import google.generativeai as genai
        genai.configure(api_key=os.environ['GEMINI_API_KEY'])
        model = genai.GenerativeModel(GEMINI_MODEL, system_instruction=system)
        resp = model.generate_content(
            user,
            generation_config={'temperature': 0, 'response_mime_type': 'application/json'},
        )
        return _parse_json_answer(resp.text)

    # mock backend — deterministic, no API key, just cites first 2 hits
    def _mock(user_text: str) -> dict:
        # pull chunk_ids out of the CONTEXT
        ids = re.findall(r'chunk_id=([A-Za-z0-9\-]+)', user_text)
        items = re.findall(r'item=(Item \S+)', user_text)
        cites = [{'chunk_id': cid, 'item': it} for cid, it in list(zip(ids, items))[:2]]
        return {
            'answer'   : '[MOCK] No LLM backend configured. This stub proves the pipeline; '
                         'set LLM_BACKEND to "openai" or "gemini" for real answers.',
            'citations': cites,
            'refused'  : False,
        }
    return _mock(user)


### 4.3 End-to-end `rag_answer()`


In [ ]:
def rag_answer(question: str,
               allowed_items: list[str] | None = None,
               k: int = TOP_K) -> dict:
    hits = retrieve(question, allowed_items=allowed_items, k=k)
    # refusal short-circuit: no hits OR best score is below threshold
    if not hits or hits[0]['score'] < MIN_SIM:
        return {
            'question' : question,
            'answer'   : 'Retrieved evidence below similarity threshold — unable to answer.',
            'citations': [],
            'refused'  : True,
            'retrieved': hits,
        }
    system, user = build_prompt(question, hits)
    out = llm_generate(system, user)
    out.setdefault('citations', [])
    out.setdefault('refused', False)
    out['question']  = question
    out['retrieved'] = hits  # attach for eval
    return out


### 4.4 Run the 15-question domain pack


In [ ]:
QUESTION_PACK_PATH = f'{OUTPUTS_DIR}/domain_questions.csv'
ANSWERS_PATH       = f'{OUTPUTS_DIR}/rag_answers.jsonl'

# If the user has uploaded questions, use them; otherwise load defaults.
DEFAULT_QUESTIONS = [
    # --- Financial Performance (Items 6, 7, 8) ---
    ('Q01', 'What are the primary drivers of revenue growth or decline reported by the company in fiscal 2024?'),
    ('Q02', 'How does management attribute margin pressure — to labor costs, inflation, supply chain, or macro factors?'),
    ('Q03', 'What does management say about liquidity and capital resources heading into the next fiscal year?'),
    ('Q04', 'Does the company report material changes in segment revenue mix?'),
    ('Q05', 'How is cash flow from operations reconciled to net income?'),
    ('Q06', 'What capital expenditure categories are disclosed, and are any linked to technology or AI investment?'),
    # --- Market Risk (Item 7A) ---
    ('Q07', 'What interest-rate risk exposure is disclosed, and how is it quantified (VaR, sensitivity, hedge notional)?'),
    ('Q08', 'What foreign-exchange risk is disclosed and which currencies are named?'),
    ('Q09', 'Is commodity or input-cost exposure discussed, and are derivative instruments used to hedge it?'),
    ('Q10', 'How does management characterize equity-market or counterparty credit risk?'),
    # --- Financial Statements footnotes (Item 8) ---
    ('Q11', 'What revenue recognition policy is described in the financial-statement footnotes?'),
    ('Q12', 'Does the company report goodwill or intangible-asset impairment in fiscal 2024? If so, how much?'),
    ('Q13', 'What are the stated income tax rate and the reconciling items to the statutory rate?'),
    ('Q14', 'What subsequent events are disclosed after the balance-sheet date?'),
    ('Q15', 'Does the auditor flag any critical audit matters, and what do they relate to?'),
]
pd.DataFrame(DEFAULT_QUESTIONS, columns=['qid','question']).to_csv(
    QUESTION_PACK_PATH, index=False)
print(f'Wrote {QUESTION_PACK_PATH}')

# Run all questions across all firms in the corpus
companies = sorted({r.get('company') for r in SECTION_ROWS if r.get('company')})
print(f'{len(companies)} firms in scope — running {len(DEFAULT_QUESTIONS)} questions each.')

answers = []
for qid, q in tqdm(DEFAULT_QUESTIONS, desc='RAG generate'):
    for company in companies:
        scoped_q = f'{q} (Firm: {company})'
        out = rag_answer(scoped_q, allowed_items=ALLOWED_ITEMS)
        out['qid']     = qid
        out['company'] = company
        answers.append(out)

with open(ANSWERS_PATH, 'w') as f:
    for a in answers:
        # drop the heavy 'retrieved' field when persisting
        a_min = {k: v for k, v in a.items() if k != 'retrieved'}
        a_min['retrieved_chunk_ids'] = [h['chunk_id'] for h in a.get('retrieved', [])]
        f.write(json.dumps(a_min) + '\n')
print(f'Wrote {len(answers)} answers → {ANSWERS_PATH}')


---
## Milestone 5 · Evaluation & System Validation

Four axes, per the rubric:
1. **Retrieval accuracy** — Recall@5 / Hit@5 on 20 labeled (question, gold chunk_id) pairs.
2. **Grounding reliability** — fraction of answers citing ≥1 chunk (citation coverage); hallucination rate = fraction of cited chunks that do NOT textually support the claim.
3. **Section integrity** — sweep all generated answers; verify every cited `item` ∈ `ALLOWED_ITEMS`.
4. **System reflection** — architecture diagram, summary table, failure-mode notes.


### 5.1 Load human-labeled eval pairs from `data/labeled_pairs.csv`

**Important rigor note.** Earlier versions of this notebook auto-populated the
gold chunk_ids by taking the first chunk that matched the gold Item — that is
circular (measuring whether the system ranks highly its own first hit).

This version loads human-labeled pairs from `data/labeled_pairs.csv`. Follow
`docs/LABELING_GUIDE.md` to fill in `gold_company` and `gold_chunk_id_contains`
before reporting final Hit@k numbers. If the CSV is not yet fully labeled, the
cell below **falls back** to the auto-populated ground truth so the notebook
still executes end-to-end — but prints a prominent WARNING so graders see the
caveat.


In [ ]:
import os
LABELED_CSV = 'data/labeled_pairs.csv'           # git-tracked, human-labeled
LABELED_OUT = f'{OUTPUTS_DIR}/labeled_pairs.csv' # what the eval loop reads

# Read the human-labeling worksheet if it exists in the repo; otherwise fall
# back to the in-notebook starter template (auto-populated, less trustworthy).
if os.path.exists(LABELED_CSV):
    labeled_df = pd.read_csv(LABELED_CSV)
    human_rows = labeled_df.dropna(subset=['gold_chunk_id_contains'])
    human_rows = human_rows[human_rows['gold_chunk_id_contains'].astype(str).str.strip() != '']
    if len(human_rows) >= 10:
        labeled = human_rows.to_dict(orient='records')
        print(f'✓ Loaded {len(labeled)} HUMAN-LABELED pairs from {LABELED_CSV}')
    else:
        print(f'⚠️  WARNING: {LABELED_CSV} exists but only {len(human_rows)} rows are labeled.')
        print('    Falling back to auto-populated ground truth for notebook execution.')
        print('    ACTION REQUIRED: label ≥20 rows per docs/LABELING_GUIDE.md before reporting final metrics.')
        labeled = None
else:
    print(f'⚠️  {LABELED_CSV} not found — using auto-populated starter labels.')
    labeled = None

# Fallback: auto-populate from the first chunk that matches each gold_item.
if labeled is None:
    _example_template = [
        {'qid':'L01','question':'liquidity and capital resources position','gold_item':'Item 7'},
        {'qid':'L02','question':'interest rate risk sensitivity analysis','gold_item':'Item 7A'},
        {'qid':'L03','question':'revenue recognition accounting policy','gold_item':'Item 8'},
        {'qid':'L04','question':'foreign currency exchange rate hedging','gold_item':'Item 7A'},
        {'qid':'L05','question':'goodwill impairment testing and result','gold_item':'Item 8'},
        {'qid':'L06','question':'operating cash flow reconciliation to net income','gold_item':'Item 8'},
        {'qid':'L07','question':'segment reporting and revenue by geography','gold_item':'Item 7'},
        {'qid':'L08','question':'commodity price hedging instruments','gold_item':'Item 7A'},
        {'qid':'L09','question':'stock based compensation expense','gold_item':'Item 8'},
        {'qid':'L10','question':'income tax rate reconciliation to statutory','gold_item':'Item 8'},
        {'qid':'L11','question':'critical accounting estimates disclosure','gold_item':'Item 7'},
        {'qid':'L12','question':'debt maturities and contractual obligations','gold_item':'Item 7'},
        {'qid':'L13','question':'revenue growth drivers year over year','gold_item':'Item 7'},
        {'qid':'L14','question':'gross margin trend and cost of revenue drivers','gold_item':'Item 7'},
        {'qid':'L15','question':'derivatives and risk management instruments','gold_item':'Item 7A'},
        {'qid':'L16','question':'deferred revenue and contract liabilities','gold_item':'Item 8'},
        {'qid':'L17','question':'allowance for credit losses methodology','gold_item':'Item 8'},
        {'qid':'L18','question':'operating expenses breakdown research development','gold_item':'Item 7'},
        {'qid':'L19','question':'counterparty credit risk exposure','gold_item':'Item 7A'},
        {'qid':'L20','question':'subsequent events after balance sheet date','gold_item':'Item 8'},
    ]
    labeled = []
    for row in _example_template:
        cand = df_chunks[df_chunks['item'] == row['gold_item']]
        if cand.empty:
            continue
        ex = cand.iloc[0]
        labeled.append({**row, 'gold_company': ex['company'],
                        'gold_chunk_id_contains': ex['chunk_id']})

pd.DataFrame(labeled).to_csv(LABELED_OUT, index=False)
print(f'Wrote {len(labeled)} labeled pairs → {LABELED_OUT}')


### 5.2 Retrieval accuracy: Recall@k / Hit@k


In [ ]:
def eval_retrieval(labeled: list[dict], k: int = TOP_K) -> pd.DataFrame:
    rows = []
    for row in labeled:
        hits = retrieve(row['question'], allowed_items=[row['gold_item']], k=k)
        hit  = any(row['gold_chunk_id_contains'] in h['chunk_id'] for h in hits)
        # recall@k here: binary (1 gold chunk per question) — Hit@k == Recall@k
        rows.append({
            'qid'       : row['qid'],
            'question'  : row['question'],
            'gold_item' : row['gold_item'],
            'hit_at_k'  : int(hit),
            'top1_chunk': hits[0]['chunk_id'] if hits else None,
            'top1_score': hits[0]['score']    if hits else None,
        })
    return pd.DataFrame(rows)

df_ret = eval_retrieval(labeled, k=TOP_K)
print(f'Hit@{TOP_K} = Recall@{TOP_K} = {df_ret.hit_at_k.mean():.1%}  '
      f'({df_ret.hit_at_k.sum()}/{len(df_ret)})')
display(df_ret.head(10))


### 5.3 Citation coverage + cross-item leakage check


In [ ]:
def eval_grounding(answers: list[dict]) -> dict:
    total, with_cites, leakers = 0, 0, 0
    leak_rows = []
    for a in answers:
        if a.get('refused'):
            continue
        total += 1
        cites = a.get('citations', [])
        if cites:
            with_cites += 1
        for c in cites:
            if c.get('item') not in ALLOWED_ITEMS:
                leakers += 1
                leak_rows.append({'qid': a.get('qid'), 'item_cited': c.get('item')})
    return {
        'n_non_refused'      : total,
        'citation_coverage'  : with_cites / total if total else float('nan'),
        'cross_item_leakers' : leakers,
        'leak_rows'          : leak_rows[:20],
    }

grounding = eval_grounding(answers)
print(f"Citation coverage : {grounding['citation_coverage']:.1%}  "
      f"(answers with ≥1 citation / answered)")
print(f"Cross-item leakage: {grounding['cross_item_leakers']}  (MUST be 0)")
if grounding['leak_rows']:
    display(pd.DataFrame(grounding['leak_rows']))


### 5.3b Real cross-domain leakage probe

The probes in §3.4 test queries with **no** in-scope answer (trivial refusal).
The real test is a query that has **both** an in-scope and an out-of-scope
natural answer — if the hard filter ever relaxes, citations would leak to
the out-of-scope Item.


In [ ]:
# Stress probe: this query is answerable from Item 7A (derivative program)
# AND from Item 10 (board risk oversight). The allow-list must suppress the
# Item 10 hits even though semantically they also match.
STRESS_PROBES = [
    ('derivative program oversight and risk committee review',
     ['Item 7A', 'Item 10']),   # natural in-scope | out-of-scope split
    ('board review of goodwill impairment testing',
     ['Item 7', 'Item 8', 'Item 10']),
    ('audit committee discussion of revenue recognition policy',
     ['Item 7', 'Item 8', 'Item 10']),
]

all_passed = True
for q, natural_items in STRESS_PROBES:
    hits = retrieve(q, allowed_items=ALLOWED_ITEMS, k=10)
    actual_items = {h['item'] for h in hits}
    leak = actual_items - set(ALLOWED_ITEMS)
    status = 'PASS' if not leak else 'FAIL'
    if leak:
        all_passed = False
    print(f'[{status}] "{q[:60]}"')
    print(f'        natural_items={natural_items}')
    print(f'        retrieved_items={actual_items}')
    print(f'        leak={leak or "none"}')

assert all_passed, 'Cross-domain leakage detected — hard filter is broken.'
print('\n✓ Cross-domain leakage probe passed — hard filter working as designed.')


### 5.4 Answer faithfulness — manual review (replaces LLM-as-judge)

LLM-as-judge using the **same backend** that generated the answers is circular.
This cell instead produces a printable worksheet: 20 (answer, cited chunks)
pairs where a human labeler marks each as *supported* or *unsupported*.

Run this cell to print the worksheet, do the review in a Google Doc or
spreadsheet, then paste the supported-count into the cell below to compute
the supported-claim rate with a 95% binomial confidence interval.


In [ ]:
import random
random.seed(42)

# Build the review sample: non-refused answers that have ≥1 citation.
_elig = [a for a in answers if not a.get('refused') and a.get('citations')]
n_review = min(20, len(_elig))
_sample = random.sample(_elig, n_review)

print(f'=== MANUAL REVIEW WORKSHEET — {n_review} answers ===')
print('For each item, mark S (supported) if every factual claim in the ANSWER')
print('is directly stated or clearly implied by at least one CITED chunk.')
print('Otherwise mark U (unsupported). Record the count below.')
print()

for i, a in enumerate(_sample, 1):
    print(f'--- #{i}  qid={a.get("qid")}  firm={a.get("company")} ---')
    print(f'QUESTION: {a.get("question")}')
    print(f'ANSWER:   {a.get("answer")}')
    for c in a.get('citations', []):
        cid = c.get('chunk_id')
        chunk = CHUNK_BY_ID.get(cid, {})
        snippet = (chunk.get('text', '') or '')[:300].replace('\n', ' ')
        print(f'  CITED {cid} [{c.get("item")}]: {snippet}…')
    print('  S / U : ___')
    print()

print('=== After review, fill in the next cell with your supported count. ===')


Fill in the count of *supported* answers from the review above, then run
the cell to compute the supported-claim rate and its 95% confidence interval.


In [ ]:
# ---- FILL IN AFTER REVIEW ----
SUPPORTED_COUNT = None   # e.g. 17 if 17/20 answers were fully supported
# -------------------------------

if SUPPORTED_COUNT is None:
    print('SUPPORTED_COUNT is None — run the worksheet above and fill in the value.')
else:
    from scipy.stats import beta
    n = len(_sample)
    k = SUPPORTED_COUNT
    rate = k / n
    # Clopper–Pearson exact 95% CI (narrow enough for small n)
    lo = beta.ppf(0.025, k,     n - k + 1) if k > 0 else 0.0
    hi = beta.ppf(0.975, k + 1, n - k    ) if k < n else 1.0
    print(f'Supported-claim rate = {rate:.1%}  ({k}/{n})')
    print(f'95% CI (Clopper–Pearson) = [{lo:.1%}, {hi:.1%}]')
    print(f'Hallucination rate ≈ {1-rate:.1%}')
    print()
    print('Pass/fail vs docs/EVALUATION_PLAN.md target (≥80% supported):',
          "PASS" if lo >= 0.80 or rate >= 0.80 else "NEEDS WORK")


### 5.5 System architecture diagram


```mermaid
flowchart TD
    A[Google Drive: SEC-10K-2024-HTML] --> B[M1: HTML parse + Item 6/7/7A/8 extraction]
    B --> C[M1: Clean: TOC bleed, signatures, boilerplate, tables→placeholder]
    C --> D[M2: Token-aware chunking ≈500 tok, 50 overlap]
    D --> E[M2: Attach metadata<br/>cik · company · ticker · year · item · chunk_id · token_count]
    E --> F[M3: bge-small-en-v1.5 embeddings<br/>384-dim, cosine-normalized]
    F --> G[M3: FAISS IndexFlatIP]
    H[User question] --> I[M3: Query embed]
    I --> G
    G --> J[M3: Filter candidates by<br/>item ∈ Item 6/7/7A/8]
    J --> K[M4: Assemble context<br/>with chunk_id+item markers]
    K --> L[M4: LLM with strict JSON contract<br/>answer · citations · refused]
    L --> M[M5: Eval — Recall@k, citation coverage,<br/>LLM-judge hallucination, leakage=0]
```


### 5.5 Ablation sweep — CHUNK_TOKENS × MIN_SIM

Justifies the chosen defaults (`CHUNK_TOKENS=500`, `MIN_SIM=0.20`) by sweeping
the 3×3 grid and reporting Hit@5 + refusal rate at each cell. Skipped by
default (set `ABLATION_MODE=True` to run — takes ~5–10 minutes).


In [ ]:
ABLATION_MODE = False   # flip to True to run the sweep

if not ABLATION_MODE:
    print('Ablation sweep skipped (set ABLATION_MODE=True to run).')
else:
    grid_chunk  = [256, 500, 1000]
    grid_minsim = [0.10, 0.20, 0.30]

    def _refusal_rate(threshold):
        refusals = 0
        for row in labeled:
            hits = retrieve(row['question'], allowed_items=[row['gold_item']], k=TOP_K)
            if not hits or hits[0]['score'] < threshold:
                refusals += 1
        return refusals / len(labeled)

    def _hit_at_k(threshold):
        rows = []
        for row in labeled:
            hits = retrieve(row['question'], allowed_items=[row['gold_item']], k=TOP_K)
            # apply the candidate MIN_SIM: treat below-threshold as refused = miss
            if hits and hits[0]['score'] < threshold:
                rows.append(0)
                continue
            rows.append(int(any(row['gold_chunk_id_contains'] in h['chunk_id'] for h in hits)))
        return sum(rows) / len(rows) if rows else float('nan')

    grid = []
    # Note: varying CHUNK_TOKENS requires a full re-chunk + re-embed — expensive.
    # For MIN_SIM, we can sweep purely at retrieval time on the CURRENT chunking.
    # This cell sweeps MIN_SIM only; to sweep CHUNK_TOKENS re-run the notebook
    # with different CHUNK_TOKENS values and append rows to this table.
    for ms in grid_minsim:
        grid.append({
            'chunk_tokens' : CHUNK_TOKENS,
            'min_sim'      : ms,
            'hit_at_5'     : _hit_at_k(ms),
            'refusal_rate' : _refusal_rate(ms),
        })

    df_ablation = pd.DataFrame(grid)
    print('Ablation sweep (MIN_SIM only — CHUNK_TOKENS fixed at current value):')
    display(df_ablation)
    df_ablation.to_csv(f'{OUTPUTS_DIR}/ablation_sweep.csv', index=False)
    print(f'Saved → {OUTPUTS_DIR}/ablation_sweep.csv')
    print('\nTo sweep CHUNK_TOKENS too, re-run the notebook for each value in',
          grid_chunk, 'and concatenate the CSVs.')


### 5.6 Performance summary


In [ ]:
summary = {
    'corpus': {
        'filings'        : len(FILING_PATHS),
        'section_rows'   : len(SECTION_ROWS),
        'chunks'         : len(CHUNKS),
        'firms'          : len({r.get('company') for r in SECTION_ROWS if r.get('company')}),
        'items_in_scope' : ALLOWED_ITEMS,
    },
    'retrieval': {
        'model'       : EMBED_MODEL,
        'k'           : TOP_K,
        'hit_at_k'    : float(df_ret.hit_at_k.mean()) if len(df_ret) else None,
        'n_labeled'   : int(len(df_ret)),
    },
    'grounding': {
        'citation_coverage'  : grounding['citation_coverage'],
        'cross_item_leakage' : grounding['cross_item_leakers'],
        'total_answers'      : len(answers),
        'refused_answers'    : sum(1 for a in answers if a.get('refused')),
    },
    'config': {
        'chunk_tokens' : CHUNK_TOKENS,
        'chunk_overlap': CHUNK_OVERLAP,
        'min_sim'      : MIN_SIM,
        'llm_backend'  : LLM_BACKEND,
    },
}
with open(f'{OUTPUTS_DIR}/performance_summary.json','w') as f:
    json.dump(summary, f, indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))


### 5.7 Reflection on limitations & failure modes

**Known failure modes observed and why they happen**

- **TOC bleed**: When a filing's table of contents uses the same `Item X.` format as real headings, a naive regex catches both. Mitigation in `filter_toc_hits()`: we require ≥400 characters between consecutive item hits, which robustly drops TOC lines but can fail for very short real items ("Not applicable.").
- **Cross-item content drift**: MD&A (Item 7) occasionally references material that also appears verbatim in Item 8 footnotes. Our strict section filter correctly refuses to pull from Item 8 when scoped to Item 7 — but that means some answers are thinner than they could be. This is **by design** (section integrity > recall).
- **Financial-table content loss**: Replacing `<table>` with `[TABLE]` placeholder trades retrievability for signal-to-noise. The current system is strong on narrative financial analysis and weak on precise numeric lookups. A follow-up iteration should index table HTML separately with a structured-table embedder.
- **Parametric-knowledge leak in M4**: LLMs sometimes answer from training-knowledge even with strict prompts. Mitigation: low temperature (0), strict JSON contract, refusal rule, and the judge-based hallucination spot-check. A stronger mitigation is constrained decoding or model-side grounding APIs (e.g., Gemini grounding, OpenAI `response_format=json_schema`).
- **Ground-truth labels are bootstrap-quality**: Section 5.1 auto-populates gold chunks from the first matching Item. Real evaluation requires a human-labeled `labeled_pairs.csv` — that file is written so the team can edit it and re-run 5.2.

**What the team should iterate on next**
- Replace auto-populated gold labels with 20+ human-labeled Q→chunk pairs per domain question.
- Add a second-stage reranker (e.g., `BAAI/bge-reranker-base`) and measure Recall@5 lift.
- Index Item-8 tables with a structured-table retriever (e.g., table-transformer or TAPAS).
- Track retrieval-score distributions over refused vs answered questions to tune `MIN_SIM`.


---
### 5.8 Deliverable index

| File (in `AD698_outputs/`) | What it is |
|---|---|
| `domain_questions.csv` | The 15 structured domain questions (M4 input). |
| `rag_answers.jsonl` | Generated answers with citations for every (question, firm) pair. |
| `labeled_pairs.csv` | 20 Q → gold-chunk pairs used for Hit@k. Edit for real labels. |
| `performance_summary.json` | Corpus stats + retrieval/grounding metrics. |

Cache artifacts in `AD698_cache/`: `sections.jsonl`, `chunks.jsonl`, `embeddings.npy`, `chunk_meta.jsonl`, `faiss.index`.

---
**End of notebook.** For the 25-pt bonus, wrap `rag_answer()` in a Streamlit app (see README).
